In [79]:
import os
import numpy as np
import pandas
from pathlib import Path
from glob import glob
import toml
from aniposelib.cameras import CameraGroup
from aniposelib.utils import load_pose2d_fnames

from hand_tracker.anipose_yt.triangulate import triangulate
from hand_tracker.triangulation.lp2anipose import lp2anipose
from hand_tracker.anipose_yt.triangulate import process_trial as triangulate_process

In [ ]:
# Load labeled frames
# Triangulate
# Reproject


## Load labeled data

In [80]:
project_dir = Path('/home/yiting/Documents/GitHub/lightning-pose/data/multiview_multical_2025')
subject = 'Neo'
trial = '2025-12-10_concat'
camera = 'camTR'
camera_idx = 4
trial_folder = f'{subject}_{trial}_{camera}'
frame_file = 'frame014530.png'
frame_file_path = project_dir / 'labeled-data' / trial_folder / frame_file
label_csv_path = project_dir / f'CollectedData_{camera}.csv'
label_df = pandas.read_csv(label_csv_path, header=[0, 1, 2])

In [81]:
# Get keypoint position from labeled data
row_idx = 197
label_frame_name = label_df.iloc[row_idx,0]
print(label_frame_name)

labeled-data/Neo_2025-12-10_concat_camTR/frame014530.png


In [82]:
import yaml
lp_config_path = '/home/yiting/Documents/GitHub/lightning-pose/scripts/configs/config_multiview_cal.yaml'

with open(lp_config_path, 'r') as file:
    lp_config = yaml.safe_load(file)

kpt_names = lp_config['data']['keypoint_names']
print(kpt_names)

['Small_Tip', 'Small_DIP', 'Small_PIP', 'Small_MCP', 'Ring_Tip', 'Ring_DIP', 'Ring_PIP', 'Ring_MCP', 'Middle_Tip', 'Middle_DIP', 'Middle_PIP', 'Middle_MCP', 'Index_Tip', 'Index_DIP', 'Index_PIP', 'Index_MCP', 'Thumb_Tip', 'Thumb_IP', 'Thumb_MCP', 'Thumb_CMC', 'Palm', 'Wrist_U', 'Wrist_R', 'Dot_t1', 'Dot_t2', 'Dot_t3', 'Dot_b1', 'Dot_b2', 'Dot_b3', 'Dot_l1', 'Dot_l2', 'Dot_l3', 'Dot_r1', 'Dot_r2', 'Dot_r3']


## Triangulation and reprojection

project(points)
Given an Nx3 array of points, this returns an CxNx2 array of 2D points, where C is the number of cameras

triangulate(points, undistort=True, progress=False)
Given an CxNx2 array, this returns an Nx3 array of points, where N is the number of points and C is the number of cameras

In [83]:
camera_views = ["To", "TL", "TR", "BL", "BR"]
debug_dir = '/home/yiting/Documents/LP_debug/20260423'

fname_dict = {}
for c in camera_views:
    lp_file_path =  os.path.join(debug_dir, 'litpose_labeled', f'CollectedData_cam{c}.csv')
    anipose_file_path = os.path.join(debug_dir, 'anipose_labeled', f'CollectedData_cam{c}.h5')
    lp2anipose(lp_file_path, anipose_file_path)
    fname_dict[c] = anipose_file_path

In [84]:
cgroup = CameraGroup.load('/home/yiting/Documents/LP_debug/20260415/calibration/calibration_results/calibration.toml')
d = load_pose2d_fnames(fname_dict, cam_names=cgroup.get_names())
n_cams, n_points, n_joints, _ = d['points'].shape
points = d['points']

bodyparts = d['bodyparts']


points_flat = points.reshape(n_cams, -1, 2)

p3ds_flat = cgroup.triangulate(points_flat, progress=True)
reprojerr_flat = cgroup.reprojection_error(p3ds_flat, points_flat, mean=True)
reproj_flat = cgroup.project(p3ds_flat)

p3ds = p3ds_flat.reshape(n_points, n_joints, 3)
reprojerr = reprojerr_flat.reshape(n_points, n_joints)
reproj = reproj_flat.reshape(n_cams, n_points, n_joints, 2)


100%|█████████████████████████| 13300/13300 [00:02<00:00, 6412.72it/s]


## Visualization

In [85]:
import cv2
img = cv2.imread(frame_file_path)

In [86]:
# Iterate and draw keypoints
for kpt in kpt_names:
    ## Get the x and y coordinates from labeled data
    x_label = label_df.iloc[row_idx, label_df.columns.get_loc(('Scorer', kpt, 'x'))]  # x coordinate from labeled data
    y_label = label_df.iloc[row_idx, label_df.columns.get_loc(('Scorer', kpt, 'y'))]  # y coordinate from labeled data

    if np.isnan(x_label) or np.isnan(y_label):
        continue  # Skip if the keypoint is not detected
    center = (int(x_label), int(y_label))

    # Draw a circle 
    cv2.circle(img, center, radius=5, color=(0, 0, 255), thickness=-1)
    

    ## Get the x and y coordinates from reprojection
    x_reproj = reproj[camera_idx, row_idx, kpt_names.index(kpt), 0]  # x coordinate from reprojection
    y_reproj = reproj[camera_idx, row_idx, kpt_names.index(kpt), 1]  # y coordinate from reprojection

    # Convert to integer coordinates for OpenCV
    if np.isnan(x_reproj) or np.isnan(y_reproj):
        continue  # Skip if the keypoint is not detected
    center = (int(x_reproj), int(y_reproj))
    
    # Draw a circle 
    cv2.circle(img, center, radius=5, color=(0, 255, 0), thickness=-1)
    


    # # Optional: Add label
    # cv2.putText(img, bp, (int(x_reproj) + 10, int(y_reproj) + 10), 
    #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

# 4. Show or Save
# cv2.imshow('Keypoint Overlay', img)
save_path = os.path.join(debug_dir, f'{subject}_{trial}_{camera}.jpg')
cv2.imwrite(save_path, img)

True